# Deliverable 2: launch and verify the interface in Colab
This companion notebook does not retrain the experiment. Upload the project ZIP, launch the interface, then generate an authentic screenshot. Original training and outputs are preserved in `AML_GNN_Real_Data_Setup.ipynb`.


In [1]:
from google.colab import files
from pathlib import Path
import zipfile, sys
uploaded = files.upload()  # Choose AML_GNN_Deliverable_2.zip
zip_name = next(name for name in uploaded if name.endswith(".zip"))
DEST = Path("/content/aml_deliverable2")
DEST.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as archive:
    for name in archive.namelist():
        path = Path(name)
        assert not path.is_absolute() and ".." not in path.parts
    archive.extractall(DEST)
ROOT = DEST / "aml-gnn"
assert (ROOT / "ui/app.py").is_file()
sys.path.insert(0, str(ROOT))
print("Project extracted:", ROOT)


Saving AML_GNN_Deliverable_2.zip to AML_GNN_Deliverable_2.zip
Project extracted: /content/aml_deliverable2/aml-gnn


In [2]:
from src.service import replay, live
result = replay("XGBoost", "recency", 49, 10)
print("VERIFIED SAVED-PREDICTION REPLAY")
print(result["metrics"])
print("LIVE SYNTHETIC PROBE (not an Elliptic transaction)")
print(live([0.0] * 93))


VERIFIED SAVED-PREDICTION REPLAY
{'mode': 'saved prediction replay; no model refit', 'model': 'XGBoost', 'policy': 'recency', 'seed': 0, 'time_step': 49, 'labeled': 476, 'illicit': 56, 'average_precision': 0.9360173635220791, 'precision': 0.9285714285714286, 'recall': 0.6964285714285714, 'f1': 0.7959183673469388, 'review_k': 48, 'precision_at_k': 0.9166666666666666, 'recall_at_k': 0.7857142857142857, 'scores_are_calibrated_probabilities': False, 'warning': 'Research only. Metrics apply to known-label transactions; no automatic sanctions.'}
LIVE SYNTHETIC PROBE (not an Elliptic transaction)
{'mode': 'live model inference; frozen XGBoost, seed 0', 'score': 0.0017565463203936815, 'threshold': 0.7, 'predicted_label': 0, 'display_label': 'below review threshold', 'input_features': 93, 'model_sha256': '2011deb6017ed159f9b65d2c592507646112ca0e07b10cf2a16b9e9009eff23c', 'engine': 'native XGBoost CPU inference', 'note': 'Weighted-model risk score, not calibrated probability or proof of illegal 

## Native-model parity on real features (recommended before submission)
The web interface uses native XGBoost when installed. A restricted numeric JSON-tree evaluator supports the lightweight installation. This cell checks both engines against actual saved predictions. It requires your original three CSVs and does not fit any model.


In [3]:
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/AML_GNN_Project/data")
# Change DATA_DIR if your original CSVs are elsewhere.
import numpy as np
import pandas as pd
import xgboost as xgb
from src.data import load_elliptic
from src.inference import PortableXGBoost
nodes, edges = load_elliptic(DATA_DIR)
columns = [c for c in nodes if c.startswith("local_")]
saved = pd.read_csv(ROOT / "results/xgb_frozen_test_predictions.csv")
sample = saved.head(50)
x = nodes.set_index("txId").loc[sample.txId, columns].to_numpy(dtype=np.float32)
booster = xgb.Booster()
booster.load_model(str(ROOT / "results/xgb_frozen_seed0.json"))
booster.set_param({"device": "cpu"})
native = booster.predict(xgb.DMatrix(x, feature_names=columns))
portable = PortableXGBoost(ROOT / "results/xgb_frozen_seed0.json").predict(x)
print("Max native vs saved difference:", np.max(np.abs(native - sample.score.to_numpy())))
print("Max portable vs native difference:", np.max(np.abs(portable - native)))
np.testing.assert_allclose(native, sample.score.to_numpy(), atol=1e-5, rtol=1e-5)
np.testing.assert_allclose(portable, native, atol=1e-5, rtol=1e-5)
real_input = x[0].tolist()
print("COPY THIS REAL FEATURE ARRAY INTO THE LIVE INFERENCE TAB:")
print(real_input)
print("Live output:", live(real_input))
print("PASS: actual-feature inference and stored prediction agree.")


Mounted at /content/drive
Max native vs saved difference: 2.9347991925554595e-08
Max portable vs native difference: 5.973785044410818e-08
COPY THIS REAL FEATURE ARRAY INTO THE LIVE INFERENCE TAB:
[-0.172982320189476, -0.05524241179227829, -1.201368808746338, -0.12196960300207138, -0.024024521932005882, -0.11300200968980789, -0.06158379465341568, -0.16364163160324097, -0.16945621371269226, -0.04970696568489075, -0.16594527661800385, -0.02874128520488739, -0.03539055213332176, -0.02271064557135105, -0.013281614519655704, -0.05740121006965637, -0.17115019261837006, -0.1729079782962799, -0.1762639582157135, -0.24329964816570282, -0.24345165491104126, -0.1386035978794098, -0.1481923758983612, -0.08014726638793945, -0.1547059565782547, -0.010763009078800678, -0.01210745144635439, -0.13862450420856476, -0.1481999158859253, -0.08014673739671707, -0.1547221541404724, -0.010668560862541199, -0.012005181983113289, 0.0016826388891786337, -0.006585183087736368, -0.02304515615105629, -0.000533490092

In [4]:
import subprocess, time, urllib.request
from google.colab import output
# Run once. If port 8000 is already occupied by this app, reuse the existing server.
try:
    urllib.request.urlopen("http://127.0.0.1:8000/", timeout=2)
    print("Using existing server on port 8000.")
except Exception:
    log_file = open(ROOT / "ui_server.log", "w")
    server = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "ui.app:app", "--host", "0.0.0.0", "--port", "8000"],
        cwd=ROOT, stdout=log_file, stderr=subprocess.STDOUT,
    )
    for attempt in range(30):
        try:
            urllib.request.urlopen("http://127.0.0.1:8000/", timeout=1)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError((ROOT / "ui_server.log").read_text())
output.serve_kernel_port_as_iframe(8000, height=1100)


<IPython.core.display.Javascript object>

## Capture your submission evidence
1. Explorer: select XGBoost, recency, step 49, 10%; click Update analysis. Capture the controls, metrics and queue.
2. Live inference: paste the real feature array from the parity cell and click Generate model prediction. Capture the input/output. The zero-vector button is only a labeled software probe.
3. Save screenshots as `docs/interface_explorer.png` and `docs/interface_live_real.png` in your local project. Commit them and your executed `ui_demo.ipynb` to GitHub.
4. Send the screenshots and the parity output back for the final report update.

The report currently uses tested API sample outputs and a labeled design diagram. It does not pretend that a browser screenshot has already been captured.
